In [1]:
%pip -q install gradio cmake ninja pyngrok

In [2]:
!nvidia-smi

Wed Oct 28 11:36:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["CUDACXX"] = "/usr/local/cuda/bin/nvcc"
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
os.environ["FORCE_CMAKE"] = "1"

# Install llama-cpp-python using the pre-compiled CUDA wheels and 2 cores for compilation
%pip install --only-binary=:all: --no-cache-dir \
  llama-cpp-python \
  --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu130 \
  --compile --config-settings=--global-option="--parallel=2"

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu130
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.8/782.8 MB 34.0 MB/s  0:00:21
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [llama-cpp-python]


In [4]:
from google.colab import drive
import os
import sqlite3

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/LocalLLM"
MODEL_PATH = os.path.join(
    BASE_DIR,
    "models",
    "qwen3-27b",
    "Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-IQ2_M.gguf",
)
CHAT_DB = os.path.join(BASE_DIR, "chat_history.sqlite3")

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found in Google Drive: {MODEL_PATH}")
if not os.path.isfile(CHAT_DB):
    raise FileNotFoundError(f"Chat database not found in Google Drive: {CHAT_DB}")

with sqlite3.connect(CHAT_DB) as con:
    con.execute("""
        CREATE TABLE IF NOT EXISTS messages (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            conversation_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT NOT NULL,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    """)

print("Model file:", MODEL_PATH)
print("Chat database:", CHAT_DB)

Mounted at /content/drive
Model file: /content/drive/MyDrive/LocalLLM/models/qwen3-27b/Qwen3.8-27B-Uncensored-HauhauCS-Aggressive-IQ2_M.gguf
Chat database: /content/drive/MyDrive/LocalLLM/chat_history.sqlite3


In [5]:

import sqlite3
import uuid
import gradio as gr

SYSTEM_PROMPT = (
    "You are a helpful AI assistant. "
    "Give clear, accurate and thoughtful answers."
)

# ---------------- Database helpers ----------------

def get_conversations():
    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT conversation_id, MAX(id) AS last_id
            FROM messages
            GROUP BY conversation_id
            ORDER BY last_id DESC
        """).fetchall()
    return [row[0] for row in rows]


def get_messages(cid):
    if not cid:
        return []

    with sqlite3.connect(CHAT_DB) as con:
        rows = con.execute("""
            SELECT role, content
            FROM messages
            WHERE conversation_id = ?
            ORDER BY id
        """, (cid,)).fetchall()

    return [
        {"role": role, "content": content}
        for role, content in rows
        if role in ("user", "assistant")
    ]


def save_message(cid, role, content):
    with sqlite3.connect(CHAT_DB) as con:
        con.execute("""
            INSERT INTO messages (conversation_id, role, content)
            VALUES (?, ?, ?)
        """, (cid, role, content))


# ---------------- Chat actions ----------------

def new_conversation():
    cid = str(uuid.uuid4())[:8]
    choices = get_conversations()

    if cid not in choices:
        choices.insert(0, cid)

    return cid, [], gr.update(choices=choices, value=cid)


def load_conversation(cid):
    if not cid:
        return "", []
    return cid, get_messages(cid)


def chat(message, cid):
    message = (message or "").strip()
    cid = (cid or "").strip()

    if not message:
        return "", get_messages(cid), cid, gr.update(
            choices=get_conversations(), value=cid or None
        )

    if not cid:
        cid = str(uuid.uuid4())[:8]

    save_message(cid, "user", message)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ] + get_messages(cid)

    try:
        result = llm.create_chat_completion(
            messages=messages,
            temperature=0.7
        )

        answer = (
            result["choices"][0]["message"].get("content") or ""
        ).strip()

        if not answer:
            answer = "(The model returned an empty response.)"

        save_message(cid, "assistant", answer)

    except Exception as e:
        print("Model inference error:", repr(e))
        answer = f"Model error: {type(e).__name__}: {e}"

    choices = get_conversations()
    if cid not in choices:
        choices.insert(0, cid)

    return (
        "",
        get_messages(cid),
        cid,
        gr.update(choices=choices, value=cid)
    )


# ---------------- Gradio interface ----------------

with gr.Blocks(title="My AI Assistant") as demo:
    gr.Markdown("# My AI Assistant")
    gr.Markdown("Qwen3.8-27B · Chat history saved in SQLite")

    with gr.Row():
        cid = gr.Textbox(
            label="Conversation ID",
            placeholder="Create a new chat or select a saved one",
            scale=3
        )
        new_btn = gr.Button("New chat")

    saved_chats = gr.Dropdown(
        label="Saved conversations",
        choices=get_conversations(),
        interactive=True
    )

    chatbot_ui = gr.Chatbot(
    height=500,
    label="Chat"
    )

    message = gr.Textbox(
        label="Your message",
        placeholder="Message your AI...",
        lines=2
    )

    send_btn = gr.Button("Send", variant="primary")

    new_btn.click(
        new_conversation,
        outputs=[cid, chatbot_ui, saved_chats]
    )

    saved_chats.change(
        load_conversation,
        inputs=saved_chats,
        outputs=[cid, chatbot_ui]
    )

    send_btn.click(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    message.submit(
        chat,
        inputs=[message, cid],
        outputs=[message, chatbot_ui, cid, saved_chats]
    )

    demo.load(
        lambda: ("", []),
        outputs=[cid, chatbot_ui]
    )

print("Chat functions restored:", callable(chat))
print("Gradio interface restored:", demo is not None)


Chat functions restored: True
Gradio interface restored: True


In [6]:
import llama_cpp
from llama_cpp import Llama

if not llama_cpp.llama_supports_gpu_offload():
    raise RuntimeError(
        "llama-cpp-python was installed without CUDA support. "
        "Re-run the CUDA installation cell in a GPU-enabled Colab runtime."
    )

llm = Llama(
    model_path=MODEL_PATH,
    n_gpu_layers=-1,
    n_ctx=18432,
    verbose=False,
)
print("Qwen model loaded with GPU offload enabled.")

ggml_cuda_init: found 1 CUDA devices (Total VRAM: 14912 MiB):
  Device 0: Tesla T4, compute capability 7.5, VMM: yes, VRAM: 14912 MiB


Qwen model loaded with GPU offload enabled.


In [11]:
import os, torch

# ---- 1. GPU memory (NVML = same numbers as nvidia-smi) ----
try:
    import pynvml
    pynvml.nvmlInit()
    info = pynvml.nvmlDeviceGetMemoryInfo(pynvml.nvmlDeviceGetHandleByIndex(0))
    vram_total, vram_used, vram_free = (x / 1024**2 for x in (info.total, info.used, info.free))
except Exception:
    free_b, total_b = torch.cuda.mem_get_info(0)
    vram_total, vram_free = total_b / 1024**2, free_b / 1024**2
    vram_used = vram_total - vram_free

# ---- 2. Model file size ----
model_size_mb = os.path.getsize(MODEL_PATH) / 1024**2 if os.path.exists(MODEL_PATH) else 0

# ---- 3. Current context ----
n_ctx = llm.n_ctx() if callable(llm.n_ctx) else llm.n_ctx

# ---- 4. Real KV cache cost from GGUF metadata ----
meta = llm.metadata
arch = meta["general.architecture"]
g = lambda k, d=None: meta.get(f"{arch}.{k}", d)

n_layers = int(g("block_count"))
n_head   = int(g("attention.head_count"))
n_kv     = g("attention.head_count_kv", n_head)
emb      = int(g("embedding_length"))
key_len  = int(g("attention.key_length", emb // n_head))
val_len  = int(g("attention.value_length", key_len))

# Number of KV heads summed over layers (handles per-layer lists and hybrid models)
if isinstance(n_kv, (list, tuple)) or hasattr(n_kv, "__len__") and not isinstance(n_kv, (str, int)):
    kv_heads_total = sum(int(x) for x in n_kv)
else:
    n_kv = int(n_kv)
    interval = g("full_attention_interval")          # hybrid models only
    attn_layers = n_layers // int(interval) if interval else n_layers
    kv_heads_total = n_kv * attn_layers

elems_per_token = kv_heads_total * (key_len + val_len)
bytes_per_token = {"f16": elems_per_token * 2.0, "q8_0": elems_per_token * 34/32}

kv_now_mb = {k: v * n_ctx / 1024**2 for k, v in bytes_per_token.items()}

# ---- 5. How much context could fit? ----
# If the model is already loaded, "free" is what's left AFTER the current KV cache.
reserve_mb = 600   # compute buffers, CUDA overhead, safety margin
extra_tokens = {k: max(0, (vram_free - reserve_mb) * 1024**2 / v) for k, v in bytes_per_token.items()}
max_ctx = {k: int(n_ctx + extra_tokens[k]) for k in bytes_per_token}

print("========== VRAM ANALYSIS ==========")
print(f"Total / Used / Free VRAM  : {vram_total:,.0f} / {vram_used:,.0f} / {vram_free:,.0f} MB")
print(f"Model file size           : {model_size_mb:,.0f} MB")
print("-----------------------------------")
print(f"Arch: {arch} | layers={n_layers} | kv_heads_total={kv_heads_total} | k/v dim={key_len}/{val_len}")
print(f"KV cost per token         : f16 {bytes_per_token['f16']/1024:,.1f} KB | q8_0 {bytes_per_token['q8_0']/1024:,.1f} KB")
print(f"KV cache at n_ctx={n_ctx:,}  : f16 {kv_now_mb['f16']:,.0f} MB | q8_0 {kv_now_mb['q8_0']:,.0f} MB")
print("-----------------------------------")
print(f"Max context that should fit (keeping {reserve_mb} MB spare):")
print(f"  with f16 KV cache       : ~{max_ctx['f16']:,} tokens")
print(f"  with q8_0 KV cache      : ~{max_ctx['q8_0']:,} tokens")
print("===================================")

========== VRAM ANALYSIS ==========
Total / Used / Free VRAM  : 15,360 / 12,104 / 3,256 MB
Model file size           : 9,842 MB
-----------------------------------
Arch: qwen35 | layers=65 | kv_heads_total=64 | k/v dim=256/256
KV cost per token         : f16 64.0 KB | q8_0 34.0 KB
KV cache at n_ctx=18,432  : f16 1,152 MB | q8_0 612 MB
-----------------------------------
Max context that should fit (keeping 600 MB spare):
  with f16 KV cache       : ~60,925 tokens
  with q8_0 KV cache      : ~98,418 tokens


In [9]:

import os
import time
import socket
from pyngrok import ngrok
from google.colab import userdata

# 1. Read Colab Secrets
ngrok_token = userdata.get("NGROK_AUTHTOKEN")
ngrok_domain = userdata.get("NGROK_DOMAIN")

if not ngrok_token or not ngrok_domain:
    raise RuntimeError(
        "Add NGROK_AUTHTOKEN and NGROK_DOMAIN to Colab Secrets."
    )

# Normalize domain
ngrok_domain = ngrok_domain.strip().rstrip("/")
ngrok_domain = (
    ngrok_domain
    .removeprefix("https://")
    .removeprefix("http://")
    .split("/")[0]
)

# 2. Configure authentication
try:
    ngrok.set_auth_token(ngrok_token)
except Exception as e:
    if "Authtoken already set in configuration file" not in str(e):
        raise

print("Ngrok authentication ready.")

# 3. Check that your existing chatbot interface exists
if "demo" not in globals():
    raise RuntimeError(
        "The variable 'demo' does not exist. Run the cell that "
        "defines your Gradio chatbot interface first."
    )

if "llm" not in globals():
    raise RuntimeError(
        "The variable 'llm' does not exist. Run your model-loading "
        "cell first; do not reinstall packages unnecessarily."
    )

# 4. Start Gradio on a free port
sock = socket.socket()
sock.bind(("127.0.0.1", 0))
port = sock.getsockname()[1]
sock.close()

print("Starting Gradio on port:", port)

demo.launch(
    server_name="0.0.0.0",
    server_port=port,
    share=False,
    prevent_thread_lock=True,
    theme=gr.themes.Soft()
)

# 5. Wait for Gradio to become available
for _ in range(30):
    try:
        with socket.create_connection(
            ("127.0.0.1", port), timeout=2
        ):
            print("Gradio is ready.")
            break
    except OSError:
        time.sleep(1)
else:
    raise RuntimeError(
        "Gradio did not start. Check the preceding launch output."
    )

# 6. Connect ngrok to the reserved domain
# Stop here with a clear error if the endpoint API rejects the config.
try:
    tunnel = ngrok.connect(
        addr=port,
        proto="http",
        url="https://" + ngrok_domain
    )

    print("\n" + "=" * 50)
    print("CHATBOT ONLINE")
    print("Public URL:", tunnel.public_url)
    print("=" * 50)

except Exception as e:
    print("Ngrok tunnel failed:", repr(e))
    print(
        "Gradio is running locally, but the public tunnel "
        "still needs to be configured."
    )
    raise


Ngrok authentication ready.
Starting Gradio on port: 36827
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Gradio is ready.

CHATBOT ONLINE
Public URL: https://cherub-yo-yo-daffodil.ngrok-free.dev


In [10]:
print("Model loaded:", "llm" in globals())
print("Chat functions loaded:", "chat" in globals())
print("Database path defined:", "CHAT_DB" in globals())
print("Gradio interface defined:", "demo" in globals())

Model loaded: True
Chat functions loaded: True
Database path defined: True
Gradio interface defined: True
